# JS14 Praktikum 1 - Klasifikasi Cat vs Dog

Dataset berasal dari tautan Google Drive pada modul JS14 dan telah diekstrak ke folder `JS14/dataset/dataset`. Notebook ini mencoba implementasi CNN Keras sesuai modul. Jika TensorFlow tidak tersedia pada environment Python 3.14, notebook menjalankan fallback klasifikasi citra yang runnable menggunakan fitur citra grayscale dan MLP.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier

DATASET_ROOT = Path("JS14/dataset/dataset")
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path("dataset/dataset")
if not DATASET_ROOT.exists():
    raise FileNotFoundError("Folder dataset tidak ditemukan.")

TRAIN_ROOT = DATASET_ROOT / "training_set"
TEST_ROOT = DATASET_ROOT / "test_set"
SINGLE_ROOT = DATASET_ROOT / "single_prediction"
class_names = ["cats", "dogs"]

counts = {
    "Training cats": len(list((TRAIN_ROOT / "cats").glob("*.jpg"))),
    "Training dogs": len(list((TRAIN_ROOT / "dogs").glob("*.jpg"))),
    "Testing cats": len(list((TEST_ROOT / "cats").glob("*.jpg"))),
    "Testing dogs": len(list((TEST_ROOT / "dogs").glob("*.jpg"))),
}
print(f"Lokasi dataset: {DATASET_ROOT.resolve()}")
print(pd.Series(counts))
print(f"Total gambar: {sum(counts.values())}")

try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    tensorflow_available = True
    print(f"TensorFlow tersedia: {tf.__version__}")
except (ImportError, ModuleNotFoundError):
    tensorflow_available = False
    print("TensorFlow tidak tersedia; menggunakan fallback MLP pada cell berikutnya.")

Lokasi dataset: C:\ListPc\244107020089_MachineLearning\JS14\dataset\dataset
Training cats    4000
Training dogs    4000
Testing cats     1000
Testing dogs     1000
dtype: int64
Total gambar: 10000
TensorFlow tidak tersedia; menggunakan fallback MLP pada cell berikutnya.


## Pra-pemrosesan dan model CNN

Pada jalur Keras, citra dinormalisasi, di-augment dengan shear, zoom, dan horizontal flip, kemudian diproses oleh CNN dua convolution-pooling layer dan dua fully connected layer.

In [2]:
if tensorflow_available:
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
    )
    test_datagen = ImageDataGenerator(rescale=1.0 / 255)
    training_set = train_datagen.flow_from_directory(
        str(TRAIN_ROOT), target_size=(64, 64), batch_size=32, class_mode="binary"
    )
    test_set = test_datagen.flow_from_directory(
        str(TEST_ROOT), target_size=(64, 64), batch_size=32,
        class_mode="binary", shuffle=False
    )

    cnn = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(64, 64, 3)),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPool2D(pool_size=2, strides=2),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPool2D(pool_size=2, strides=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    start = time.perf_counter()
    history = cnn.fit(training_set, validation_data=test_set, epochs=5)
    training_time = time.perf_counter() - start
    loss, accuracy = cnn.evaluate(test_set, verbose=0)
    print(f"Akurasi testing: {accuracy:.4f}")
    print(f"Loss testing: {loss:.4f}")
    print(f"Waktu training: {training_time:.2f} detik")
else:
    def load_images(root, image_size=(32, 32)):
        images, labels = [], []
        for label, class_name in enumerate(class_names):
            for path in sorted((root / class_name).glob("*.jpg")):
                with Image.open(path) as image:
                    image = image.convert("L").resize(image_size)
                    images.append(np.asarray(image, dtype=np.float32).reshape(-1) / 255.0)
                labels.append(label)
        return np.asarray(images), np.asarray(labels)

    X_train, y_train = load_images(TRAIN_ROOT)
    X_test, y_test = load_images(TEST_ROOT)
    fallback_model = MLPClassifier(
        hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
        batch_size=64, max_iter=25, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=5, random_state=42,
    )
    start = time.perf_counter()
    fallback_model.fit(X_train, y_train)
    training_time = time.perf_counter() - start
    predictions = fallback_model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Data training: {X_train.shape}")
    print(f"Data testing: {X_test.shape}")
    print(f"Akurasi testing fallback: {accuracy:.4f}")
    print(f"Loss akhir fallback: {fallback_model.loss_:.6f}")
    print(f"Waktu training: {training_time:.2f} detik")

Data training: (8000, 1024)
Data testing: (2000, 1024)
Akurasi testing fallback: 0.5995
Loss akhir fallback: 0.645454
Waktu training: 3.71 detik


## Evaluasi dan prediksi satu citra

In [3]:
single_images = sorted(SINGLE_ROOT.glob("*.jpg"))
print("Contoh file prediksi:", [path.name for path in single_images])

if tensorflow_available:
    test_set.reset()
    probabilities = cnn.predict(test_set, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)
    y_test = test_set.classes
    print(classification_report(y_test, predictions, target_names=["cats", "dogs"], zero_division=0))
    for path in single_images:
        image = tf.keras.utils.load_img(path, target_size=(64, 64))
        array = tf.keras.utils.img_to_array(image) / 255.0
        probability = float(cnn.predict(np.expand_dims(array, axis=0), verbose=0)[0][0])
        print(f"{path.name}: {class_names[int(probability >= 0.5)]} ({probability:.4f})")
else:
    print(classification_report(y_test, predictions, target_names=class_names, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))
    for path in single_images:
        with Image.open(path) as image:
            array = np.asarray(image.convert("L").resize((32, 32)), dtype=np.float32)
        probability = float(fallback_model.predict_proba(array.reshape(1, -1) / 255.0)[0, 1])
        print(f"{path.name}: {class_names[int(probability >= 0.5)]} ({probability:.4f})")

Contoh file prediksi: ['cat_or_dog_1.jpg', 'cat_or_dog_2.jpg']
              precision    recall  f1-score   support

        cats       0.60      0.62      0.61      1000
        dogs       0.60      0.58      0.59      1000

    accuracy                           0.60      2000
   macro avg       0.60      0.60      0.60      2000
weighted avg       0.60      0.60      0.60      2000

Confusion matrix:
[[619 381]
 [420 580]]
cat_or_dog_1.jpg: dogs (0.5317)
cat_or_dog_2.jpg: dogs (0.6709)


## Kesimpulan

CNN pada jalur Keras melakukan ekstraksi fitur citra secara otomatis melalui convolution dan pooling. Hasil evaluasi perlu dilihat bersama loss, akurasi, confusion matrix, dan contoh prediksi satu citra. Jika TensorFlow belum tersedia, fallback tetap memproses dataset yang sama untuk verifikasi data dan klasifikasi dasar, tetapi bukan pengganti arsitektur CNN.